# 🤖 Notebook 3 — Modélisation & Comparaison des Modèles
---
**Objectif** : Entraîner et comparer 8 modèles de régression en appliquant les stratégies avancées pour atteindre R² ≥ 0.90.

**Stratégies appliquées :**
- Variable cible : log(1 + prix) pour normaliser la distribution asymétrique
- TargetEncoder sklearn pour les catégorielles (leave-one-out, sans data leakage)
- 30 features enrichies : distances géographiques, features texte, Target Encoding ville
- Pipeline sklearn complet : imputation → encodage → normalisation → modèle
- Validation croisée 5-fold sur tous les modèles

**Modèles évalués :**
Baseline → Linéaire → Ridge → Lasso → Decision Tree → Random Forest → Gradient Boosting → XGBoost

## Table des matières
1. Chargement et configuration
2. Pipeline de preprocessing avancé
3. Fonctions d'évaluation
4. Baseline — DummyRegressor
5. Régression Linéaire
6. Ridge
7. Lasso
8. Decision Tree
9. Random Forest
10. Gradient Boosting
11. XGBoost
12. Comparaison finale


## 1. Chargement et configuration

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings, json, time
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"]    = (13, 5)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing   import StandardScaler, TargetEncoder
from sklearn.impute          import SimpleImputer
from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy           import DummyRegressor
from sklearn.linear_model    import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor

try:
    from xgboost import XGBRegressor
    XGBOOST_OK = True
    print("XGBoost disponible.")
except ImportError:
    XGBOOST_OK = False
    print("XGBoost absent. Installer avec : pip install xgboost")

print("Librairies ML chargees.")


In [ ]:
df = pd.read_csv("dataset_final.csv")
with open("features_config.json") as f:
    cfg = json.load(f)

NUMERIC_FEATURES     = cfg["NUMERIC_FEATURES"]
CATEGORICAL_FEATURES = cfg["CATEGORICAL_FEATURES"]
TARGET               = cfg["TARGET"]
LOG_TARGET           = cfg.get("LOG_TARGET", "log_price")

df        = df.dropna(subset=[TARGET])
X         = df[[c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES if c in df.columns]]
y         = df[TARGET]
y_log     = np.log1p(y)

num_feats = [f for f in NUMERIC_FEATURES     if f in X.columns]
cat_feats = [f for f in CATEGORICAL_FEATURES if f in X.columns]

X_train, X_test,  y_train,  y_test   = train_test_split(X, y,     test_size=0.2, random_state=42)
_,       _,       yl_train, yl_test   = train_test_split(X, y_log, test_size=0.2, random_state=42)

print(f"Train : {len(X_train):,}  |  Test : {len(X_test):,}")
print(f"Prix median train    : {y_train.median():,.0f} FCFA")
print(f"log(prix) moyen      : {yl_train.mean():.4f}  +-  {yl_train.std():.4f}")


## 2. Pipeline de preprocessing avancé

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute        import SimpleImputer

def build_pipeline(model, use_log=True):
    """
    Construit un pipeline sklearn complet.

    Traitement numerique  : imputation par la mediane, puis standardisation.
    Traitement categoriel : imputation par constante, puis TargetEncoder
                            (encode chaque modalite par le prix moyen,
                             avec leave-one-out pour eviter le data leakage).
    """
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="inconnu")),
        ("encoder", TargetEncoder(target_type="continuous", smooth="auto")),
    ])
    prep = ColumnTransformer([
        ("num", num_pipe, num_feats),
        ("cat", cat_pipe, cat_feats),
    ])
    return Pipeline([("prep", prep), ("model", model)])

print("Pipeline configure :")
print("  Numeriques    : SimpleImputer(median)   -> StandardScaler")
print("  Categorielles : SimpleImputer(constant) -> TargetEncoder (leave-one-out)")
print("  Variable cible: log(1 + prix) pour tous les modeles")


## 3. Fonctions d'évaluation

In [ ]:
RESULTS = {}

def evaluate(name, pipe, use_log=True, cv=5):
    """
    Entraine, evalue et stocke les resultats d'un modele.
    Toutes les metriques finales sont exprimees en FCFA reels.
    """
    y_tr = yl_train if use_log else y_train
    y_te = yl_test  if use_log else y_test

    t0 = time.time()
    pipe.fit(X_train, y_tr)
    elapsed = time.time() - t0

    yp_tr = pipe.predict(X_train)
    yp_te = pipe.predict(X_test)

    if use_log:
        yp_tr_r, yp_te_r = np.expm1(yp_tr), np.expm1(yp_te)
        y_tr_r,  y_te_r  = np.expm1(y_tr.values), np.expm1(y_te.values)
    else:
        yp_tr_r, yp_te_r = yp_tr, yp_te
        y_tr_r,  y_te_r  = y_tr.values, y_te.values

    r2_te = r2_score(y_te_r, yp_te_r)
    r2_tr = r2_score(y_tr_r, yp_tr_r)
    mae   = mean_absolute_error(y_te_r, yp_te_r)
    rmse  = np.sqrt(mean_squared_error(y_te_r, yp_te_r))
    mape  = np.mean(np.abs((y_te_r - yp_te_r) / (y_te_r + 1))) * 100

    kf   = KFold(n_splits=cv, shuffle=True, random_state=42)
    cv_s = cross_val_score(pipe, X_train, y_tr, cv=kf, scoring="r2", n_jobs=-1)

    RESULTS[name] = {
        "R2 test":  round(r2_te, 4),
        "R2 train": round(r2_tr, 4),
        "MAE":      round(mae, 2),
        "RMSE":     round(rmse, 2),
        "MAPE":     round(mape, 2),
        "CV R2":    round(cv_s.mean(), 4),
        "CV std":   round(cv_s.std(), 4),
        "Overfit":  round(r2_tr - r2_te, 4),
        "Temps":    round(elapsed, 1),
        "use_log":  use_log,
        "pipeline": pipe,
        "y_pred":   yp_te_r,
        "cv_scores":cv_s,
    }
    return RESULTS[name]

def show(name, res):
    g    = res["R2 train"] - res["R2 test"]
    flag = "OK" if g < 0.05 else "leger surapprentissage" if g < 0.15 else "SURAPPRENTISSAGE"
    obj  = "OBJECTIF ATTEINT" if res["R2 test"] >= 0.90 else f"manque {0.90 - res['R2 test']:.4f}"
    print(f"  R2 test    : {res['R2 test']:.4f}  [{obj}]")
    print(f"  R2 train   : {res['R2 train']:.4f}  [{flag}] (gap = {g:.4f})")
    print(f"  MAE        : {res['MAE']:>15,.0f} FCFA")
    print(f"  RMSE       : {res['RMSE']:>15,.0f} FCFA")
    print(f"  MAPE       : {res['MAPE']:.2f} %")
    print(f"  CV R2      : {res['CV R2']:.4f} +- {res['CV std']:.4f}")
    print(f"  Temps      : {res['Temps']} s")

def plot_residuals(name, ax1, ax2):
    res   = RESULTS[name]
    yp    = res["y_pred"]
    resid = y_test.values - yp
    ax1.scatter(yp / 1e6, resid / 1e6, alpha=0.3, s=6, color="#2196F3")
    ax1.axhline(0, color="red", linestyle="--", lw=1.5)
    ax1.set_xlabel("Predit (M FCFA)"); ax1.set_ylabel("Residu (M FCFA)")
    ax1.set_title(f"Residus - {name}", fontweight="bold")
    ax2.hist(resid / 1e6, bins=50, color="#4CAF50", alpha=0.7, edgecolor="white")
    ax2.axvline(0, color="red", linestyle="--", lw=1.5)
    ax2.set_xlabel("Residu (M FCFA)")
    ax2.set_title(f"Distribution residus - {name}", fontweight="bold")

print("Fonctions d'evaluation definies.")


## 4. Baseline — DummyRegressor

In [ ]:
print("=" * 60)
print("  BASELINE - DummyRegressor (mediane)")
print("=" * 60)
print()
print("Reference minimale absolue : le modele predit toujours la mediane")
print("du prix d'entrainement sans exploiter aucune feature.")
print("Tout modele utile doit obtenir un R2 significativement superieur.")
print()
r = evaluate("Baseline",
             build_pipeline(DummyRegressor(strategy="median"), use_log=False),
             use_log=False)
show("Baseline", r)
print(f"
Reference a depasser : R2 = {r['R2 test']:.4f} | MAE = {r['MAE']:,.0f} FCFA")


## 5. Régression Linéaire

In [ ]:
print("=" * 60)
print("  REGRESSION LINEAIRE - comparaison prix brut vs log(prix)")
print("=" * 60)
print()
print("On compare l'entrainement sur le prix brut versus log(prix)")
print("pour mesurer l'apport de la transformation logarithmique.")
print()

r_raw = evaluate("Lineaire (prix brut)",
                 build_pipeline(LinearRegression(), use_log=False), use_log=False)
r_log = evaluate("Lineaire (log prix)",
                 build_pipeline(LinearRegression(), use_log=True),  use_log=True)

print("Sans log(prix) :")
show("Lineaire (prix brut)", r_raw)
print("
Avec log(prix) :")
show("Lineaire (log prix)", r_log)
print(f"
Gain de R2 apporte par log : {r_log['R2 test'] - r_raw['R2 test']:+.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_residuals("Lineaire (log prix)", axes[0], axes[1])
plt.tight_layout()
plt.savefig("fig_lr_residus.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Ridge (régularisation L2)

In [ ]:
print("=" * 60)
print("  RIDGE - Regularisation L2")
print("=" * 60)
print()
print("Ridge ajoute une penalite proportionnelle a la somme des carres")
print("des coefficients. Cela reduit le surapprentissage sans eliminer")
print("de features. L'alpha optimal est selectionne par CV interne.")
print()

rc   = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100, 500, 1000, 5000], cv=5)
build_pipeline(rc, use_log=True).fit(X_train, yl_train)
best_alpha = rc.alpha_
print(f"Alpha optimal (RidgeCV) : {best_alpha}")

r = evaluate("Ridge", build_pipeline(Ridge(alpha=best_alpha)), use_log=True)
show("Ridge", r)
print(f"Gain vs Lineaire (log) : {r['R2 test'] - RESULTS['Lineaire (log prix)']['R2 test']:+.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_residuals("Ridge", axes[0], axes[1])
plt.tight_layout()
plt.savefig("fig_ridge_residus.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Lasso (régularisation L1 + sélection de features)

In [ ]:
print("=" * 60)
print("  LASSO - Regularisation L1")
print("=" * 60)
print()
print("Lasso penalise la somme des valeurs absolues des coefficients.")
print("Contrairement a Ridge, Lasso pousse certains coefficients")
print("exactement a zero, realisant ainsi une selection automatique.")
print()

lc   = LassoCV(cv=5, max_iter=10000, random_state=42)
p_l  = build_pipeline(lc, use_log=True)
p_l.fit(X_train, yl_train)
best_al   = p_l.named_steps["model"].alpha_
coefs     = p_l.named_steps["model"].coef_
n_nonzero = (coefs != 0).sum()
print(f"Alpha optimal (LassoCV)   : {best_al:.6f}")
print(f"Features selectionnees    : {n_nonzero} / {len(coefs)}")
print(f"Features eliminees (= 0)  : {len(coefs) - n_nonzero}")

r = evaluate("Lasso",
             build_pipeline(Lasso(alpha=best_al, max_iter=10000)), use_log=True)
show("Lasso", r)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_residuals("Lasso", axes[0], axes[1])
plt.tight_layout()
plt.savefig("fig_lasso_residus.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Decision Tree

In [ ]:
print("=" * 60)
print("  DECISION TREE")
print("=" * 60)
print()
print("L'arbre segmente recursivement l'espace des features.")
print("On teste plusieurs profondeurs pour identifier le meilleur")
print("compromis biais-variance et visualiser le surapprentissage.")
print()

depths = [3, 5, 8, 10, 15, None]
r2_tr_d, r2_te_d = [], []

print(f"  {'Profondeur':<12} {'R2 train':>10} {'R2 test':>10} {'Gap':>10}")
print("  " + "-" * 44)
for d in depths:
    p  = build_pipeline(DecisionTreeRegressor(max_depth=d, random_state=42), use_log=True)
    p.fit(X_train, yl_train)
    rt = r2_score(y_train.values, np.expm1(p.predict(X_train)))
    re = r2_score(y_test.values,  np.expm1(p.predict(X_test)))
    r2_tr_d.append(rt); r2_te_d.append(re)
    print(f"  {str(d):<12} {rt:>10.4f} {re:>10.4f} {rt-re:>10.4f}")

best_d = depths[np.argmax(r2_te_d)]
print(f"
Meilleure profondeur : {best_d}")

r = evaluate("Decision Tree",
             build_pipeline(DecisionTreeRegressor(max_depth=best_d, random_state=42)),
             use_log=True)
show("Decision Tree", r)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dl = [str(d) if d else "None" for d in depths]
axes[0].plot(dl, r2_tr_d, "o-", color="#2196F3", label="Train", lw=2)
axes[0].plot(dl, r2_te_d, "s-", color="#4CAF50", label="Test",  lw=2)
axes[0].set_xlabel("Profondeur"); axes[0].set_ylabel("R2")
axes[0].set_title("R2 selon la profondeur - Decision Tree", fontweight="bold")
axes[0].legend(); axes[0].grid(True, alpha=0.3)
plot_residuals("Decision Tree", axes[1], axes[1])
plt.tight_layout()
plt.savefig("fig_dt.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Random Forest

In [ ]:
print("=" * 60)
print("  RANDOM FOREST")
print("=" * 60)
print()
print("Le Random Forest agree N arbres entrained sur des sous-echantillons.")
print("L'agregation reduit la variance et ameliore la generalisation.")
print("On teste n_estimators pour trouver le point de convergence optimal.")
print()

ns  = [50, 100, 200, 300, 500]
r2s = []
print("Test du nombre d'arbres :")
for n in ns:
    p  = build_pipeline(RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1), use_log=True)
    p.fit(X_train, yl_train)
    re = r2_score(y_test.values, np.expm1(p.predict(X_test)))
    r2s.append(re)
    print(f"  n = {n:>3}  ->  R2 test = {re:.4f}")

best_n = ns[np.argmax(r2s)]
print(f"
Meilleur n_estimators : {best_n}")

r = evaluate("Random Forest",
             build_pipeline(RandomForestRegressor(
                 n_estimators=best_n, max_features="sqrt",
                 random_state=42, n_jobs=-1)),
             use_log=True)
show("Random Forest", r)

imp    = r["pipeline"].named_steps["model"].feature_importances_
fn_imp = num_feats + [f"cat_{i}" for i in range(len(imp) - len(num_feats))]
imp_df = (pd.DataFrame({"Feature": fn_imp[:len(imp)], "Importance": imp})
          .sort_values("Importance", ascending=True).tail(20))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].barh(imp_df["Feature"], imp_df["Importance"],
             color=plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(imp_df))))
axes[0].set_title("Top 20 features - Random Forest", fontweight="bold")
axes[0].set_xlabel("Importance")
plot_residuals("Random Forest", axes[1], axes[1])
plt.tight_layout()
plt.savefig("fig_rf.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. Gradient Boosting

In [ ]:
print("=" * 60)
print("  GRADIENT BOOSTING")
print("=" * 60)
print()
print("Le Gradient Boosting construit les arbres sequentiellement.")
print("Chaque arbre corrige les erreurs residuelles du precedent.")
print("On teste plusieurs learning rates pour optimiser la convergence.")
print()

lrs = [0.03, 0.05, 0.1, 0.15]
print("Test du learning rate :")
for lr in lrs:
    p  = build_pipeline(GradientBoostingRegressor(
             n_estimators=200, learning_rate=lr, max_depth=5, random_state=42),
             use_log=True)
    p.fit(X_train, yl_train)
    re = r2_score(y_test.values, np.expm1(p.predict(X_test)))
    print(f"  lr = {lr}  ->  R2 test = {re:.4f}")

r = evaluate("Gradient Boosting",
             build_pipeline(GradientBoostingRegressor(
                 n_estimators=500, learning_rate=0.05, max_depth=5,
                 subsample=0.8, min_samples_split=5, random_state=42)),
             use_log=True)
show("Gradient Boosting", r)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_residuals("Gradient Boosting", axes[0], axes[1])
plt.tight_layout()
plt.savefig("fig_gb.png", dpi=150, bbox_inches="tight")
plt.show()


## 11. XGBoost

In [ ]:
print("=" * 60)
print("  XGBOOST")
print("=" * 60)
print()
print("XGBoost est une implementation optimisee du gradient boosting.")
print("Il integre nativement regularisation L1/L2, gestion des NaN")
print("et parallelisme. Generalement le plus performant sur donnees tabulaires.")
print()

if XGBOOST_OK:
    r = evaluate("XGBoost",
                 build_pipeline(XGBRegressor(
                     n_estimators=1000, learning_rate=0.03, max_depth=6,
                     subsample=0.8, colsample_bytree=0.8,
                     reg_alpha=0.1, reg_lambda=1.0, min_child_weight=3,
                     random_state=42, verbosity=0, n_jobs=-1)),
                 use_log=True)
    show("XGBoost", r)

    imp_x  = r["pipeline"].named_steps["model"].feature_importances_
    fn_x   = num_feats + [f"cat_{i}" for i in range(len(imp_x) - len(num_feats))]
    imp_xdf = (pd.DataFrame({"Feature": fn_x[:len(imp_x)], "Importance": imp_x})
               .sort_values("Importance", ascending=True).tail(20))

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    axes[0].barh(imp_xdf["Feature"], imp_xdf["Importance"],
                 color=plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(imp_xdf))))
    axes[0].set_title("Top 20 features - XGBoost", fontweight="bold")
    plot_residuals("XGBoost", axes[1], axes[1])
    plt.tight_layout()
    plt.savefig("fig_xgb.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("XGBoost absent. Installer avec : pip install xgboost")


## 12. Comparaison finale

In [ ]:
rows = []
for name, res in RESULTS.items():
    rows.append({
        "Modele":   name,
        "R2 test":  res["R2 test"],  "R2 train": res["R2 train"],
        "MAE":      res["MAE"],      "RMSE":     res["RMSE"],
        "MAPE (%)": res["MAPE"],
        "CV R2":    res["CV R2"],    "CV std":   res["CV std"],
        "Overfit":  res["Overfit"],  "Temps (s)":res["Temps"],
    })

mdf = (pd.DataFrame(rows)
       .sort_values("R2 test", ascending=False)
       .reset_index(drop=True))
mdf.index += 1

print("CLASSEMENT FINAL")
print(mdf.to_string())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
names  = mdf["Modele"].tolist()
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(names)))

def hbar(ax, vals, title, xlabel, div=1, inv=False):
    c    = colors[::-1] if inv else colors
    bars = ax.barh(names, [v / div for v in vals], color=c)
    ax.set_title(title, fontweight="bold"); ax.set_xlabel(xlabel)
    for bar, v in zip(bars, vals):
        ax.text(v / div * 1.01, bar.get_y() + bar.get_height() / 2,
                f"{v/div:.4f}" if div == 1 else f"{v/div:.1f}M",
                va="center", fontsize=7)

hbar(axes[0, 0], mdf["R2 test"].values,  "R2 Test (superieur = meilleur)",  "R2")
hbar(axes[0, 1], mdf["MAE"].values,      "MAE (inferieur = meilleur)",      "M FCFA", div=1e6, inv=True)
hbar(axes[1, 0], mdf["CV R2"].values,    "CV R2 5-fold (superieur = meilleur)", "CV R2")

ov     = mdf["Overfit"].values
col_ov = ["#ef5350" if v > 0.1 else "#ff9800" if v > 0.05 else "#4CAF50" for v in ov]
axes[1, 1].barh(names, ov, color=col_ov)
axes[1, 1].axvline(0.05, color="orange", linestyle="--", label="Seuil 0.05")
axes[1, 1].axvline(0.10, color="red",    linestyle="--", label="Seuil 0.10")
axes[1, 1].set_title("Overfitting gap train/test (inferieur = meilleur)", fontweight="bold")
axes[1, 1].legend(fontsize=8)

plt.suptitle("Comparaison finale - tous les modeles", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_comparaison_finale.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
medals = ["🥇", "🥈", "🥉"] + ["  "] * 20
best    = mdf.iloc[0]["Modele"]
base_r2 = RESULTS["Baseline"]["R2 test"]

print("=" * 65)
print("  PODIUM FINAL")
print("=" * 65)
for i, (_, row) in enumerate(mdf.iterrows()):
    m   = medals[i]
    g   = row["R2 test"] - base_r2
    obj = "OBJECTIF ATTEINT" if row["R2 test"] >= 0.90 else f"manque {0.90 - row['R2 test']:.4f}"
    print(f"
{m} {row['Modele']}")
    print(f"     R2 test       : {row['R2 test']:.4f}  [{obj}]")
    print(f"     Gain baseline : {g:+.4f}")
    print(f"     MAE           : {row['MAE']:,.0f} FCFA")
    print(f"     CV R2         : {row['CV R2']:.4f} +- {row['CV std']:.4f}")

print(f"
{'='*65}")
print(f"Modele selectionne pour optimisation (NB4) : {best}")

res_save = {
    name: {k: float(v) if isinstance(v, (float, int, np.floating, np.integer)) else str(v)
           for k, v in res.items() if k not in ("pipeline", "y_pred", "cv_scores")}
    for name, res in RESULTS.items()
}
res_save["best_model"] = best
with open("comparison_results.json", "w") as f:
    json.dump(res_save, f, indent=2)
print("
Resultats sauvegardes -> comparison_results.json")
